In [1]:
import os
os.chdir("..")

In [2]:
%pwd

'd:\\Projects\\RAG_Systems\\ProductDemand\\product-demand-forecasting-system'

In [3]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    target_column: str  


In [4]:
import sys
from pathlib import Path
from src.productdemand.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH, SCHEMA_FILE_PATH
from src.productdemand.utils.common import read_yaml, create_directories
from src.productdemand.exception.custom_exception import CustomException

class ConfigurationManager:
    def __init__(
            self,
            config_filepath=CONFIG_FILE_PATH,
            params_filepath=PARAMS_FILE_PATH,
            schema_filepath=SCHEMA_FILE_PATH,
        ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([Path(self.config.artifacts_root)])

    def get_model_trainer_config(self) -> ModelTrainerConfig:
        config = self.config.model_trainer
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_trainer_config = ModelTrainerConfig(
            root_dir=Path(config.root_dir),
            train_data_path=Path(config.train_data_path),
            test_data_path=Path(config.test_data_path),
            model_name=config.model_name,
            target_column=schema.name
        )

        return model_trainer_config



In [ ]:
import pandas as pd
import os
from productdemand import logger
from sklearn.linear_model import ElasticNet
from sklearn.model_selection import GridSearchCV
import joblib
from productdemand.entity.config_entity import ModelTrainerConfig

from src.productdemand.exception.custom_exception import CustomException
from src.productdemand.logger.custom_logger import get_logger

logger = get_logger(__name__)

class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train(self):
        
        train_data = pd.read_csv(self.config.train_data_path)
        test_data = pd.read_csv(self.config.test_data_path)

        
        train_x = train_data.drop([self.config.target_column], axis=1)
        test_x = test_data.drop([self.config.target_column], axis=1)
        train_y = train_data[self.config.target_column]
        test_y = test_data[self.config.target_column]

        # ۳. تعریف پارامترها برای GridSearch (می‌توانی از params.yaml بخوانی)
        param_grid = {
            'alpha': [0.1, 0.5, 1.0],
            'l1_ratio': [0.1, 0.5, 0.9]
        }

        # ۴. اجرای GridSearchCV
        logger.info("Starting GridSearch for ElasticNet...")
        enet = ElasticNet(random_state=42)
        grid_search = GridSearchCV(
            estimator=enet,
            param_grid=param_grid,
            cv=5,
            scoring='neg_mean_squared_error',
            verbose=1
        )
        
        grid_search.fit(train_x, train_y)

        # ۵. استخراج بهترین مدل
        best_model = grid_search.best_estimator_
        logger.info(f"Best parameters found: {grid_search.best_params_}")

        # ۶. ذخیره مدل نهایی
        joblib.dump(best_model, os.path.join(self.config.root_dir, self.config.model_name))
        logger.info(f"Best model saved at {self.config.root_dir}")
